In [ ]:
# # !pip install -U "triton-windows<3.8"
# # !pip uninstall torch torchvision torchaudio -y
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

^C


In [2]:
import torch
print(f"Is GPU/CUDA available? -> {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

Is GPU/CUDA available? -> True
Device Name: NVIDIA GeForce RTX 2050


In [3]:
import torch
from single_perceptron_triton import single_perceptron_kernel,run_single_perceptron
from multiple_perceptron_triton import multiple_perceptron_kernel,run_multiple_perceptron_layer
from Backpropagation_triton import run_triton_backward

In [3]:
# Single Perceptron
X_features = torch.tensor([1.0, 2.0, 1.0], device='cuda', dtype=torch.float32)
W_weights  = torch.tensor([0.5, -0.8, 1.2], device='cuda', dtype=torch.float32)
b_bias     = torch.tensor([-0.3], device='cuda', dtype=torch.float32)
perceptron_prediction = run_single_perceptron(X_features, W_weights, b_bias)

raw_score = torch.dot(X_features, W_weights) + b_bias
pytorch_reference = 1 if raw_score.item() >= 0.0 else 0

print("=" * 50)
print("   JUPYTER NOTEBOOK TRITON EXECUTION LOGS")
print("=" * 50)
print(f"Computed Linear Score (Z) : {raw_score.item():.4f}")
print(f"Triton Kernel Output      : {perceptron_prediction}")
print(f"PyTorch Reference Check   : {pytorch_reference}")
print(f"System Correctness Status : {perceptron_prediction == pytorch_reference}")
print("=" * 50)

   JUPYTER NOTEBOOK TRITON EXECUTION LOGS
Computed Linear Score (Z) : -0.2000
Triton Kernel Output      : 0
PyTorch Reference Check   : 0
System Correctness Status : True


In [ ]:
batch_size, in_features, out_features = 64, 32, 16

# Instantiating random data matrices directly on your GPU VRAM
X_matrix = torch.randn(batch_size, in_features, device='cuda')
W_matrix = torch.randn(out_features, in_features, device='cuda')
b_vector = torch.randn(out_features, device='cuda')

# Execute your custom Triton Multi-Perceptron Layer
triton_output = run_multiple_perceptron_layer(X_matrix, W_matrix, b_vector)

# Generate baseline validation check using standard PyTorch operations
pytorch_reference = torch.where((torch.matmul(X_matrix, W_matrix.t()) + b_vector) >= 0.0, 1.0, 0.0)
match_verified = torch.allclose(triton_output, pytorch_reference)

print("=" * 50)
print("   MULTI-PERCEPTRON LAYER VALIDATION LOGS")
print("=" * 50)
print(f"Input Features Data Size : {list(X_matrix.shape)}")
print(f"Output Neurons Data Size : {list(triton_output.shape)}")
print(f"Mathematical Match Status: {match_verified}")
print("=" * 50)

   MULTI-PERCEPTRON LAYER VALIDATION LOGS
Input Features Data Size : [64, 32]
Output Neurons Data Size : [64, 16]
Mathematical Match Status: True


: 

In [ ]:
batch, in_feats, out_feats = 32, 16, 8

X_tensor = torch.randn(batch, in_feats, device='cuda')
W_tensor = torch.randn(out_feats, in_feats, device='cuda', requires_grad=True)
b_tensor = torch.randn(out_feats, device='cuda', requires_grad=True)

Z_mock = torch.matmul(X_tensor, W_tensor.t()) + b_tensor
loss = Z_mock.sum()
loss.backward()  # Triggers PyTorch's native Autograd backprop calculation

dZ_incoming = Z_mock.grad if Z_mock.grad is not None else torch.ones_like(Z_mock)

triton_dW, triton_db = run_triton_backward(dZ_incoming, X_tensor)

triton_dW_contiguous = triton_dW.contiguous()
pytorch_dW_contiguous = W_tensor.grad.contiguous()

dw_match = torch.allclose(triton_dW_contiguous, pytorch_dW_contiguous, rtol=1e-2, atol=1e-2)
db_match = torch.allclose(triton_db, b_tensor.grad, rtol=1e-2, atol=1e-2)

print("=" * 50)
print("   BACKPROPAGATION GRADIENT VALIDATION LOGS")
print("=" * 50)
print(f"Weight Gradient (dW) Match: {dw_match}")
print(f"Bias Gradient (db) Match  : {db_match}")
print("=" * 50)

if not dw_match:
    print(f"Max absolute variation detected: {torch.max(torch.abs(triton_dW_contiguous - pytorch_dW_contiguous))}")

C:\Users\Ayush\AppData\Local\Temp\ipykernel_2692\2852150055.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\build\aten\src\ATen/core/TensorBody.h:494.)
  dZ_incoming = Z_mock.grad if Z_mock.grad is not None else torch.ones_like(Z_mock)


   BACKPROPAGATION GRADIENT VALIDATION LOGS
Weight Gradient (dW) Match: True
Bias Gradient (db) Match  : True
